# Web & Search Tools — Information Retrieval for Agents

## What This Notebook Teaches You

LLMs have a **knowledge cutoff** — they can't answer questions about recent events, proprietary data, or specialized domains without external information. Search tools are the **eyes** of an agent.

In this notebook, you will:

1. **Build a document corpus** — 30 diverse articles covering AI, science, history, and technology
2. **Implement TF-IDF search from scratch** — term frequency × inverse document frequency scoring
3. **Implement semantic search** — dense embeddings with FAISS for meaning-based retrieval
4. **Build content extraction tools** — extract relevant passages from long documents
5. **Create a research agent** — that searches, extracts, and synthesizes information
6. **Implement multi-query search** — reformulate queries when initial results are poor

Everything runs **locally** against our own corpus — no external APIs needed.

---

> **Prerequisites**: Notebooks 01–05 (agents, tools, ReAct). Notebook 14 (memory) recommended.
> **Runtime**: Google Colab with T4 GPU.
> **Time**: ~60 minutes.

## Part 0: Environment Setup

We load Qwen3-8B for reasoning plus BGE embeddings for semantic search.

In [ ]:
# --- Google Colab Setup ---
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch sentence-transformers faiss-cpu numpy

import torch
import time
import json
import re
import math
import numpy as np
import faiss
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Union
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer

# Mount Google Drive for model caching
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", quantization_config=quantization_config,
    cache_dir=CACHE_DIR, torch_dtype="auto",
)

# Load embedding model for memory/retrieval
embed_model = SentenceTransformer("BAAI/bge-base-en-v1.5", cache_folder=CACHE_DIR)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample, top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

def embed_texts(texts):
    """Embed a list of texts using BGE model. Returns numpy array."""
    return embed_model.encode(texts, normalize_embeddings=True)

print(f"✓ LLM loaded: {MODEL_NAME}")
print(f"  GPU memory: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"✓ Embedding model loaded: BAAI/bge-base-en-v1.5 (768-dim)")

## 1. Why Agents Need Search

### The Knowledge Gap Problem

LLMs are trained on static snapshots of the internet. They cannot:

- Answer questions about events after their training cutoff
- Access private or proprietary information
- Verify whether their "knowledge" is still accurate
- Provide citations or sources for claims

**Search tools bridge this gap** by giving agents access to external information at inference time.

### Search as a Tool

In the tool-use framework from Notebook 05, search is just another tool:

```
Agent thinks: "I need to find information about X"
Agent calls: search(query="X")  
Agent receives: [relevant documents]
Agent synthesizes: answer based on documents
```

In this notebook, we build the search infrastructure from scratch — no APIs, no black boxes.

## 2. Building a Document Corpus

We create a local knowledge base of 30 documents across four topics. Each document has a title, multi-paragraph content, and topic label. This gives us a realistic corpus to search against.

In [ ]:
# Build a diverse document corpus for search experiments

CORPUS = [
    # --- AI & Machine Learning ---
    {
        "id": 0,
        "title": "The Transformer Architecture Revolution",
        "topic": "AI",
        "content": (
            "The transformer architecture, introduced in the 2017 paper 'Attention Is All You Need' "
            "by Vaswani et al., fundamentally changed natural language processing. Unlike recurrent "
            "neural networks that process tokens sequentially, transformers use self-attention to "
            "process all tokens in parallel, enabling much faster training on modern hardware.\n\n"
            "The key innovation is the scaled dot-product attention mechanism, which computes "
            "relevance scores between all pairs of positions in a sequence. This allows the model "
            "to capture long-range dependencies without the vanishing gradient problems that "
            "plagued RNNs and LSTMs.\n\n"
            "Since 2017, transformers have become the foundation for virtually all state-of-the-art "
            "language models, including BERT, GPT, T5, and their successors. The architecture has "
            "also been adapted for computer vision (ViT), audio processing, and protein structure "
            "prediction."
        )
    },
    {
        "id": 1,
        "title": "Reinforcement Learning from Human Feedback",
        "topic": "AI",
        "content": (
            "RLHF is a technique for aligning language models with human preferences. The process "
            "involves three stages: first, supervised fine-tuning on high-quality demonstrations; "
            "second, training a reward model on human comparisons of model outputs; and third, "
            "optimizing the language model against the reward model using proximal policy optimization.\n\n"
            "The technique gained prominence when OpenAI used it to train InstructGPT and later "
            "ChatGPT. The reward model learns to predict which of two responses a human would "
            "prefer, capturing nuanced quality judgments that are difficult to specify with rules.\n\n"
            "Recent variations include Direct Preference Optimization (DPO), which eliminates the "
            "separate reward model by directly optimizing the language model on preference pairs. "
            "Constitutional AI uses AI feedback instead of human feedback for some alignment steps."
        )
    },
    {
        "id": 2,
        "title": "Neural Network Scaling Laws",
        "topic": "AI",
        "content": (
            "Research by Kaplan et al. at OpenAI revealed that language model performance follows "
            "predictable power laws as model size, dataset size, and compute budget increase. "
            "These scaling laws suggest that simply making models bigger and training them on more "
            "data leads to reliable improvements in capability.\n\n"
            "The Chinchilla paper by Hoffmann et al. refined these laws, showing that many models "
            "were undertrained relative to their size. The optimal allocation of a fixed compute "
            "budget involves training a smaller model on significantly more data than was common "
            "practice.\n\n"
            "These findings have profound implications for AI development strategy. Organizations "
            "must balance model size against data availability and training cost. The emergence of "
            "new capabilities at certain scale thresholds remains an active area of research."
        )
    },
    {
        "id": 3,
        "title": "Retrieval-Augmented Generation",
        "topic": "AI",
        "content": (
            "RAG combines the generative capabilities of large language models with the precision "
            "of information retrieval systems. Instead of relying solely on parametric knowledge "
            "stored in model weights, RAG retrieves relevant documents from an external knowledge "
            "base and includes them in the generation context.\n\n"
            "The original RAG paper by Lewis et al. used a dense retriever (DPR) to find relevant "
            "passages from Wikipedia, which were then concatenated with the query as input to a "
            "sequence-to-sequence model. This approach dramatically improved factual accuracy on "
            "knowledge-intensive tasks.\n\n"
            "Modern RAG systems use sophisticated chunking strategies, hybrid retrieval combining "
            "sparse and dense methods, and re-ranking stages. RAG has become the dominant approach "
            "for building knowledge-grounded applications with LLMs."
        )
    },
    {
        "id": 4,
        "title": "AI Agent Architectures",
        "topic": "AI",
        "content": (
            "AI agents are systems that perceive their environment, reason about goals, and take "
            "actions to achieve them. Modern LLM-based agents combine language model reasoning "
            "with tool use, memory, and planning capabilities to solve complex tasks autonomously.\n\n"
            "The ReAct framework interleaves reasoning traces with actions, allowing agents to "
            "think step-by-step while interacting with external tools. More advanced architectures "
            "like Reflexion add self-evaluation and memory to enable learning from mistakes.\n\n"
            "Multi-agent systems extend this further by having multiple specialized agents "
            "collaborate. Architectures range from simple pipelines to complex hierarchies where "
            "manager agents delegate to specialist workers."
        )
    },
    {
        "id": 5,
        "title": "Prompt Engineering Techniques",
        "topic": "AI",
        "content": (
            "Prompt engineering is the practice of designing inputs to language models that elicit "
            "desired outputs. Techniques range from simple instruction formatting to sophisticated "
            "multi-step reasoning frameworks.\n\n"
            "Chain-of-thought prompting, introduced by Wei et al., improves reasoning by asking "
            "the model to show its work step by step. Few-shot prompting provides examples of the "
            "desired input-output pattern. Tree-of-thought extends chain-of-thought by exploring "
            "multiple reasoning paths.\n\n"
            "Effective prompts are specific, provide context, define the output format, and include "
            "constraints. System prompts set the overall behavior, while user prompts provide the "
            "specific task. The interplay between prompt design and model capabilities is a key "
            "skill for AI practitioners."
        )
    },
    {
        "id": 6,
        "title": "Embeddings and Vector Databases",
        "topic": "AI",
        "content": (
            "Text embeddings are dense vector representations that capture semantic meaning. "
            "Similar texts produce similar vectors, enabling nearest-neighbor search to find "
            "semantically related content regardless of exact word overlap.\n\n"
            "Modern embedding models like sentence-transformers are trained with contrastive "
            "learning objectives. They learn to place similar texts close together and dissimilar "
            "texts far apart in high-dimensional space. BGE, E5, and GTE are popular embedding "
            "model families.\n\n"
            "Vector databases like FAISS, Pinecone, and Weaviate provide efficient similarity "
            "search over large collections of embeddings. They use approximate nearest neighbor "
            "algorithms like HNSW and IVF to achieve sub-linear search time."
        )
    },
    # --- Science ---
    {
        "id": 7,
        "title": "CRISPR Gene Editing Technology",
        "topic": "science",
        "content": (
            "CRISPR-Cas9 is a revolutionary gene editing technology adapted from a natural defense "
            "mechanism found in bacteria. The system uses a guide RNA to direct the Cas9 enzyme "
            "to a specific location in the genome, where it makes a precise cut in the DNA strand.\n\n"
            "The technology was developed by Jennifer Doudna and Emmanuelle Charpentier, who "
            "received the Nobel Prize in Chemistry in 2020. CRISPR enables researchers to add, "
            "remove, or modify genetic material with unprecedented precision and efficiency.\n\n"
            "Applications range from treating genetic diseases like sickle cell anemia to "
            "engineering crops with improved yields and pest resistance. Ethical debates continue "
            "about the use of CRISPR for human germline editing."
        )
    },
    {
        "id": 8,
        "title": "Quantum Computing Fundamentals",
        "topic": "science",
        "content": (
            "Quantum computers use quantum bits (qubits) that can exist in superposition — "
            "simultaneously representing both 0 and 1. This property, combined with entanglement "
            "and interference, allows quantum computers to explore many solutions in parallel.\n\n"
            "Current quantum computers are in the NISQ (Noisy Intermediate-Scale Quantum) era, "
            "with dozens to hundreds of qubits that are prone to errors. Google claimed quantum "
            "supremacy in 2019 with its 53-qubit Sycamore processor on a specific sampling task.\n\n"
            "Practical applications include drug discovery through molecular simulation, "
            "cryptography through Shor's algorithm for factoring large numbers, and optimization "
            "problems in logistics and finance. Error correction remains the major challenge."
        )
    },
    {
        "id": 9,
        "title": "mRNA Vaccine Technology",
        "topic": "science",
        "content": (
            "mRNA vaccines work by delivering genetic instructions that teach cells to produce a "
            "harmless piece of the target pathogen, triggering an immune response. The technology "
            "was decades in development before its rapid deployment against COVID-19.\n\n"
            "Key innovations include modified nucleosides that prevent the immune system from "
            "destroying the mRNA before it reaches cells, and lipid nanoparticle delivery systems "
            "that protect the fragile mRNA molecules and help them enter cells.\n\n"
            "The speed advantage of mRNA vaccines is remarkable — once the viral sequence is known, "
            "a candidate vaccine can be designed in days. Future applications include personalized "
            "cancer vaccines, treatments for rare genetic diseases, and universal flu vaccines."
        )
    },
    {
        "id": 10,
        "title": "Dark Matter and Dark Energy",
        "topic": "science",
        "content": (
            "Observations of galaxy rotation curves, gravitational lensing, and the cosmic "
            "microwave background indicate that approximately 85 percent of matter in the "
            "universe is dark matter — it interacts gravitationally but does not emit or absorb "
            "light.\n\n"
            "Dark energy is even more mysterious, making up about 68 percent of the total "
            "energy density of the universe. Its existence was inferred from the unexpected "
            "discovery in 1998 that the expansion of the universe is accelerating.\n\n"
            "Together, dark matter and dark energy constitute about 95 percent of the universe, "
            "yet their fundamental nature remains unknown. Candidate dark matter particles include "
            "WIMPs and axions. Dark energy might be Einstein's cosmological constant or a dynamic "
            "scalar field."
        )
    },
    {
        "id": 11,
        "title": "Climate Change Science",
        "topic": "science",
        "content": (
            "The scientific consensus, supported by multiple independent lines of evidence, is "
            "that human activities — primarily burning fossil fuels — have caused global average "
            "temperatures to rise by approximately 1.1 degrees Celsius since pre-industrial times.\n\n"
            "The greenhouse effect is well-understood physics: CO2 and other gases absorb and "
            "re-emit infrared radiation, trapping heat in the atmosphere. Ice cores show that "
            "current CO2 levels (over 420 ppm) are higher than at any point in the past 800,000 "
            "years.\n\n"
            "Impacts include rising sea levels, more frequent extreme weather events, ocean "
            "acidification, and biodiversity loss. The IPCC recommends limiting warming to 1.5°C "
            "to avoid the most catastrophic consequences."
        )
    },
    {
        "id": 12,
        "title": "The Human Microbiome",
        "topic": "science",
        "content": (
            "The human body hosts trillions of microorganisms — bacteria, viruses, fungi, and "
            "archaea — collectively known as the microbiome. The gut microbiome alone contains "
            "an estimated 100 trillion bacteria representing over 1,000 species.\n\n"
            "Research has revealed that the microbiome plays crucial roles in digestion, immune "
            "system development, mental health (the gut-brain axis), and disease susceptibility. "
            "Disruption of the microbiome (dysbiosis) is linked to conditions including obesity, "
            "inflammatory bowel disease, and depression.\n\n"
            "The Human Microbiome Project and MetaHIT have cataloged microbial species and genes "
            "across different body sites. Therapeutic approaches include fecal microbiota "
            "transplants, targeted probiotics, and dietary interventions."
        )
    },
    # --- History ---
    {
        "id": 13,
        "title": "The Industrial Revolution",
        "topic": "history",
        "content": (
            "The Industrial Revolution, beginning in Britain in the late 18th century, transformed "
            "human civilization from agrarian economies to industrial ones. Key innovations included "
            "the steam engine, spinning jenny, power loom, and later the railroad.\n\n"
            "The revolution occurred in two phases: the first (1760-1840) centered on textiles, "
            "steam power, and iron; the second (1870-1914) brought electricity, steel, chemicals, "
            "and petroleum. Both phases dramatically increased productivity and living standards.\n\n"
            "Social consequences were profound — urbanization, the rise of factory labor, child "
            "labor controversies, and the emergence of trade unions and labor movements. The "
            "revolution set the foundation for modern capitalism and the consumer economy."
        )
    },
    {
        "id": 14,
        "title": "The Space Race",
        "topic": "history",
        "content": (
            "The Space Race was a Cold War competition between the United States and Soviet Union "
            "for spaceflight supremacy. The Soviets struck first with Sputnik in 1957, the first "
            "artificial satellite, and Yuri Gagarin's orbital flight in 1961.\n\n"
            "President Kennedy's 1961 declaration to land a man on the Moon before the decade's "
            "end galvanized the American space program. The Apollo program consumed 4% of the "
            "federal budget at its peak, employing 400,000 people.\n\n"
            "On July 20, 1969, Neil Armstrong and Buzz Aldrin became the first humans to walk on "
            "the Moon. The achievement required innovations in rocketry, computing, materials "
            "science, and project management that continue to benefit technology today."
        )
    },
    {
        "id": 15,
        "title": "The Printing Press and Information Revolution",
        "topic": "history",
        "content": (
            "Johannes Gutenberg's movable type printing press, developed around 1440, is often "
            "cited as one of the most important inventions in human history. Before printing, "
            "books were hand-copied by scribes, making them extremely expensive and rare.\n\n"
            "The printing press reduced the cost of books by roughly 80 percent within decades. "
            "By 1500, an estimated 20 million volumes had been printed in Europe. This "
            "democratization of knowledge fueled the Renaissance, the Reformation, and the "
            "Scientific Revolution.\n\n"
            "The parallels to the modern internet are striking — both technologies dramatically "
            "reduced the cost of distributing information, disrupted existing power structures, "
            "and created new challenges around misinformation and information overload."
        )
    },
    {
        "id": 16,
        "title": "Ancient Greek Philosophy and Democracy",
        "topic": "history",
        "content": (
            "Ancient Athens, in the 5th century BCE, developed the world's first known democracy, "
            "where citizens could directly participate in governance through the Assembly. "
            "Simultaneously, Greek philosophers laid the foundations for Western thought.\n\n"
            "Socrates developed the dialectical method of inquiry through questioning. His student "
            "Plato wrote dialogues exploring justice, beauty, and the ideal state. Aristotle, "
            "Plato's student, systematized logic, ethics, politics, and natural philosophy.\n\n"
            "Greek contributions to mathematics (Euclid, Pythagoras), medicine (Hippocrates), "
            "and science (Archimedes) established methodologies still used today. The concept of "
            "rational inquiry as the path to knowledge remains the foundation of science."
        )
    },
    {
        "id": 17,
        "title": "The Silk Road Trade Network",
        "topic": "history",
        "content": (
            "The Silk Road was a network of trade routes connecting China to the Mediterranean, "
            "active from roughly the 2nd century BCE to the 15th century CE. Despite its name, "
            "silk was just one of many goods traded along its routes.\n\n"
            "Merchants transported spices, precious metals, gemstones, glass, textiles, and paper "
            "across thousands of miles of desert, mountains, and steppe. The routes facilitated "
            "not just trade but the exchange of ideas, religions, technologies, and diseases.\n\n"
            "Buddhism spread from India to China and beyond via Silk Road connections. Paper and "
            "gunpowder traveled from China westward. The Black Death likely spread along trade "
            "routes from Central Asia to Europe in the 14th century."
        )
    },
    {
        "id": 18,
        "title": "The Renaissance",
        "topic": "history",
        "content": (
            "The Renaissance, meaning 'rebirth,' was a cultural and intellectual movement that "
            "began in Italy in the 14th century and spread across Europe over the next three "
            "centuries. It marked a transition from medieval to modern ways of thinking.\n\n"
            "Key figures include Leonardo da Vinci (art, engineering, anatomy), Michelangelo "
            "(sculpture, painting, architecture), Galileo (astronomy, physics), and Machiavelli "
            "(political philosophy). The movement emphasized humanism, observation, and the "
            "rediscovery of classical Greek and Roman texts.\n\n"
            "The Renaissance transformed art (linear perspective, realism), science (empirical "
            "method), politics (secular governance), and commerce (banking, double-entry "
            "bookkeeping). Florence, under the Medici family's patronage, was its epicenter."
        )
    },
    # --- Technology ---
    {
        "id": 19,
        "title": "The Internet Protocol Suite",
        "topic": "technology",
        "content": (
            "TCP/IP is the foundational protocol suite of the internet, developed in the 1970s "
            "by Vint Cerf and Bob Kahn. The design philosophy of end-to-end connectivity and "
            "packet switching created a network that is resilient, scalable, and decentralized.\n\n"
            "The protocol stack has four layers: link (Ethernet, WiFi), internet (IP addressing "
            "and routing), transport (TCP for reliable delivery, UDP for speed), and application "
            "(HTTP, SMTP, DNS). This layered design allows each layer to evolve independently.\n\n"
            "The transition from IPv4 (4.3 billion addresses) to IPv6 (340 undecillion addresses) "
            "is ongoing. The internet now connects over 5 billion people and an estimated 15 "
            "billion IoT devices worldwide."
        )
    },
    {
        "id": 20,
        "title": "Blockchain and Distributed Ledgers",
        "topic": "technology",
        "content": (
            "Blockchain technology, introduced by the Bitcoin whitepaper in 2008, provides a "
            "decentralized, immutable ledger of transactions without requiring a trusted "
            "intermediary. Each block contains a cryptographic hash of the previous block, "
            "creating a tamper-evident chain.\n\n"
            "Consensus mechanisms like Proof of Work and Proof of Stake ensure that all "
            "participants agree on the state of the ledger. Smart contracts, popularized by "
            "Ethereum, allow programmable logic to execute automatically when conditions are met.\n\n"
            "Beyond cryptocurrency, blockchain applications include supply chain tracking, "
            "digital identity verification, decentralized finance (DeFi), and non-fungible "
            "tokens (NFTs). Energy consumption of Proof of Work systems remains controversial."
        )
    },
    {
        "id": 21,
        "title": "Cloud Computing Architecture",
        "topic": "technology",
        "content": (
            "Cloud computing delivers computing resources — servers, storage, databases, "
            "networking, software — over the internet on a pay-as-you-go basis. The three main "
            "service models are IaaS, PaaS, and SaaS, offering increasing levels of abstraction.\n\n"
            "Virtualization and containerization (Docker, Kubernetes) enable efficient resource "
            "sharing and application deployment. Microservices architecture decomposes applications "
            "into small, independently deployable services that communicate via APIs.\n\n"
            "Major providers include AWS, Azure, and Google Cloud. The shift to cloud has "
            "transformed software development, enabling rapid scaling, global deployment, and "
            "reduced capital expenditure. Edge computing extends cloud capabilities closer to "
            "data sources."
        )
    },
    {
        "id": 22,
        "title": "Cybersecurity Threats and Defenses",
        "topic": "technology",
        "content": (
            "Cybersecurity encompasses the protection of computer systems, networks, and data "
            "from digital attacks. Common threats include malware, phishing, ransomware, SQL "
            "injection, cross-site scripting, and distributed denial-of-service attacks.\n\n"
            "Defense strategies operate at multiple layers: network security (firewalls, IDS/IPS), "
            "application security (input validation, authentication), data security (encryption, "
            "access controls), and organizational security (training, incident response plans).\n\n"
            "The zero-trust security model, which assumes no user or system is inherently "
            "trustworthy, is replacing traditional perimeter-based approaches. AI is increasingly "
            "used both by attackers (automated phishing, deepfakes) and defenders (anomaly "
            "detection, threat hunting)."
        )
    },
    {
        "id": 23,
        "title": "Semiconductor Manufacturing",
        "topic": "technology",
        "content": (
            "Modern semiconductor chips are manufactured using photolithography, where patterns "
            "are projected onto silicon wafers coated with light-sensitive material. Current "
            "leading-edge processes use extreme ultraviolet (EUV) lithography at 3nm and below.\n\n"
            "TSMC, Samsung, and Intel are the only companies capable of manufacturing chips at "
            "advanced nodes. A single EUV lithography machine from ASML costs over $150 million "
            "and requires multiple 747 cargo planes to ship.\n\n"
            "Moore's Law — the observation that transistor density doubles roughly every two "
            "years — has driven exponential growth in computing power for decades. While physical "
            "limits are approaching, innovations like 3D stacking, chiplets, and new materials "
            "continue to push performance forward."
        )
    },
    {
        "id": 24,
        "title": "Open Source Software Movement",
        "topic": "technology",
        "content": (
            "The open source movement, formalized by the Open Source Initiative in 1998, promotes "
            "software whose source code is freely available for anyone to inspect, modify, and "
            "distribute. It grew from the earlier free software movement led by Richard Stallman.\n\n"
            "Linux, created by Linus Torvalds in 1991, is the most prominent open source project. "
            "It now powers the majority of servers, all of the world's top supercomputers, and "
            "most mobile devices through Android. Other major projects include Apache, PostgreSQL, "
            "Python, and Kubernetes.\n\n"
            "Open source has fundamentally changed how software is developed and distributed. "
            "Companies like Red Hat, Elastic, and MongoDB have built billion-dollar businesses "
            "around open source software through support, services, and managed offerings."
        )
    },
    {
        "id": 25,
        "title": "5G and Next-Generation Wireless",
        "topic": "technology",
        "content": (
            "5G is the fifth generation of cellular network technology, offering peak speeds up "
            "to 20 Gbps, latency under 1 millisecond, and support for up to one million devices "
            "per square kilometer. It uses three spectrum bands: low, mid, and millimeter wave.\n\n"
            "The technology enables new applications including autonomous vehicles (requiring "
            "ultra-low latency), remote surgery, industrial IoT, and augmented reality. Network "
            "slicing allows operators to create virtual networks optimized for different use cases.\n\n"
            "Deployment challenges include the need for denser infrastructure (especially for "
            "mmWave), high costs, and geopolitical concerns around equipment suppliers. Research "
            "into 6G is already underway, targeting even higher speeds and AI-native networks."
        )
    },
    {
        "id": 26,
        "title": "Electric Vehicles and Battery Technology",
        "topic": "technology",
        "content": (
            "Electric vehicles have seen rapid adoption driven by falling battery costs, "
            "government incentives, and growing environmental awareness. Lithium-ion batteries, "
            "the dominant EV technology, have dropped from over $1,000 per kWh in 2010 to under "
            "$140 per kWh in 2023.\n\n"
            "Tesla pioneered the premium EV market, while traditional automakers are now "
            "investing hundreds of billions in electrification. Key challenges include charging "
            "infrastructure, range anxiety, battery recycling, and raw material sourcing.\n\n"
            "Solid-state batteries promise higher energy density and safety compared to liquid "
            "electrolyte cells. Sodium-ion batteries offer a cheaper alternative using abundant "
            "materials. Vehicle-to-grid technology could turn EVs into distributed energy storage."
        )
    },
    # --- Additional cross-topic ---
    {
        "id": 27,
        "title": "The Ethics of Artificial Intelligence",
        "topic": "AI",
        "content": (
            "As AI systems become more capable and widely deployed, ethical concerns have moved "
            "from academic discussion to urgent policy questions. Key issues include bias in "
            "training data, lack of transparency in decision-making, privacy erosion, and the "
            "potential for job displacement.\n\n"
            "Algorithmic bias can perpetuate and amplify existing social inequalities. Studies "
            "have shown biases in facial recognition systems, hiring algorithms, and criminal "
            "justice risk assessment tools. Addressing bias requires diverse training data, "
            "careful evaluation, and ongoing monitoring.\n\n"
            "Frameworks for responsible AI development include the EU AI Act, NIST AI Risk "
            "Management Framework, and various corporate AI ethics guidelines. The tension "
            "between innovation speed and safety remains a central challenge."
        )
    },
    {
        "id": 28,
        "title": "Nuclear Fusion Research",
        "topic": "science",
        "content": (
            "Nuclear fusion — the process that powers the sun — promises virtually unlimited "
            "clean energy if it can be harnessed on Earth. Fusion combines light atomic nuclei "
            "(typically hydrogen isotopes deuterium and tritium) to release enormous energy.\n\n"
            "Two main approaches are being pursued: magnetic confinement (tokamaks like ITER) "
            "and inertial confinement (laser-driven fusion like NIF). In December 2022, the "
            "National Ignition Facility achieved fusion ignition — producing more energy from "
            "fusion than the laser energy used to initiate it.\n\n"
            "ITER, the international tokamak project in France, aims to demonstrate sustained "
            "fusion power by the 2030s. Private companies including Commonwealth Fusion Systems "
            "and TAE Technologies are pursuing alternative designs."
        )
    },
    {
        "id": 29,
        "title": "The History of Computing",
        "topic": "technology",
        "content": (
            "The history of computing spans from Charles Babbage's Analytical Engine concept "
            "in the 1830s to modern cloud and quantum computing. Key milestones include Alan "
            "Turing's theoretical foundations in the 1930s, ENIAC (1945), the transistor (1947), "
            "and the integrated circuit (1958).\n\n"
            "The personal computer revolution of the 1970s-80s, led by Apple, IBM, and Microsoft, "
            "brought computing to homes and offices. The World Wide Web, invented by Tim "
            "Berners-Lee in 1989, transformed the internet from an academic network into a "
            "global platform for commerce, communication, and knowledge.\n\n"
            "Moore's Law has driven exponential improvement in computing power, enabling "
            "smartphones that surpass the capabilities of room-sized mainframes from decades "
            "past. The current frontier includes AI accelerators, quantum processors, and "
            "neuromorphic chips."
        )
    },
]

print(f"Corpus loaded: {len(CORPUS)} documents")
topics = {}
for doc in CORPUS:
    topics[doc['topic']] = topics.get(doc['topic'], 0) + 1
for topic, count in sorted(topics.items()):
    print(f"  {topic}: {count} documents")
print(f"  Total words: {sum(len(d['content'].split()) for d in CORPUS):,}")

## 3. TF-IDF Search from Scratch

TF-IDF (Term Frequency × Inverse Document Frequency) is the foundational algorithm for text search. It scores documents based on how **frequently** query terms appear in a document (TF) weighted by how **rare** those terms are across the corpus (IDF).

### 3.1 — The Math

- **Term Frequency**: `TF(t, d) = count(t in d) / len(d)`
- **Inverse Document Frequency**: `IDF(t) = log(N / (1 + df(t)))` where `df(t)` is the number of documents containing term `t`
- **TF-IDF Score**: `TF(t, d) × IDF(t)`
- **Query Score**: Sum of TF-IDF scores for each query term

In [ ]:
import string
from collections import Counter

class TFIDFSearchEngine:
    """TF-IDF search engine built from scratch — no sklearn or external libraries."""

    def __init__(self):
        self.documents = []
        self.doc_tokens = []       # tokenized versions
        self.doc_freqs = Counter() # df(t) = number of docs containing term t
        self.num_docs = 0

    def _tokenize(self, text):
        """Lowercase, remove punctuation, split into words."""
        text = text.lower()
        text = text.translate(str.maketrans('', '', string.punctuation))
        return text.split()

    def index(self, documents):
        """Index a list of documents (each with 'title' and 'content')."""
        self.documents = documents
        self.num_docs = len(documents)
        self.doc_tokens = []

        # Tokenize all documents (combine title + content)
        for doc in documents:
            tokens = self._tokenize(doc['title'] + ' ' + doc['content'])
            self.doc_tokens.append(tokens)

        # Compute document frequencies
        self.doc_freqs = Counter()
        for tokens in self.doc_tokens:
            unique_terms = set(tokens)
            for term in unique_terms:
                self.doc_freqs[term] += 1

        print(f"Indexed {self.num_docs} documents")
        print(f"Vocabulary size: {len(self.doc_freqs):,} unique terms")

    def _tf(self, term, tokens):
        """Term frequency: count of term / total tokens in document."""
        return tokens.count(term) / len(tokens) if tokens else 0

    def _idf(self, term):
        """Inverse document frequency: log(N / (1 + df))."""
        df = self.doc_freqs.get(term, 0)
        return math.log(self.num_docs / (1 + df))

    def search(self, query, top_k=5):
        """Search the corpus and return top-k results with scores."""
        query_tokens = self._tokenize(query)

        scores = []
        for idx, doc_tokens in enumerate(self.doc_tokens):
            score = 0
            for term in query_tokens:
                tf = self._tf(term, doc_tokens)
                idf = self._idf(term)
                score += tf * idf
            scores.append((idx, score))

        # Sort by score descending
        scores.sort(key=lambda x: x[1], reverse=True)

        results = []
        for idx, score in scores[:top_k]:
            if score > 0:
                results.append({
                    "doc_id": idx,
                    "title": self.documents[idx]["title"],
                    "topic": self.documents[idx]["topic"],
                    "score": round(score, 4),
                    "preview": self.documents[idx]["content"][:150] + "..."
                })
        return results

# Build the search engine
tfidf_engine = TFIDFSearchEngine()
tfidf_engine.index(CORPUS)

### 3.2 — Testing TF-IDF Search

Let's run several queries and examine the results. TF-IDF works well for keyword-heavy queries but struggles with semantic understanding.

In [ ]:
# Test queries demonstrating TF-IDF strengths and weaknesses
test_queries = [
    "transformer architecture attention mechanism",
    "gene editing CRISPR",
    "history of space exploration",
    "how do batteries work in electric cars",
    "ethical implications of artificial intelligence",
    "what happened on the silk road",
]

for query in test_queries:
    results = tfidf_engine.search(query, top_k=3)
    print(f"\n{'='*70}")
    print(f"Query: \"{query}\"")
    print(f"{'='*70}")
    for i, r in enumerate(results):
        print(f"  {i+1}. [{r['topic']}] {r['title']} (score: {r['score']:.4f})")
    if not results:
        print("  No results found.")

## 4. Semantic Search with Embeddings

TF-IDF matches **words** — semantic search matches **meaning**. "Car battery technology" and "electric vehicle energy storage" mean similar things but share few words. Dense embeddings capture this.

### 4.1 — Building a FAISS Index

In [ ]:
class SemanticSearchEngine:
    """Semantic search using sentence embeddings and FAISS."""

    def __init__(self):
        self.documents = []
        self.index = None
        self.embeddings = None

    def build_index(self, documents):
        """Embed all documents and build FAISS index."""
        self.documents = documents

        # Create text representations for embedding
        texts = [
            f"{doc['title']}. {doc['content']}" for doc in documents
        ]

        print(f"Embedding {len(texts)} documents...")
        start = time.time()
        self.embeddings = embed_texts(texts)
        elapsed = time.time() - start
        print(f"  Embedded in {elapsed:.1f}s — shape: {self.embeddings.shape}")

        # Build FAISS index
        dim = self.embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dim)  # Inner product (cosine sim for normalized vecs)
        self.index.add(self.embeddings.astype('float32'))
        print(f"  FAISS index built: {self.index.ntotal} vectors, {dim} dimensions")

    def search(self, query, top_k=5):
        """Search by semantic similarity."""
        query_embedding = embed_texts([query]).astype('float32')
        scores, indices = self.index.search(query_embedding, top_k)

        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx >= 0:
                results.append({
                    "doc_id": int(idx),
                    "title": self.documents[idx]["title"],
                    "topic": self.documents[idx]["topic"],
                    "score": round(float(score), 4),
                    "preview": self.documents[idx]["content"][:150] + "..."
                })
        return results

# Build semantic search engine
semantic_engine = SemanticSearchEngine()
semantic_engine.build_index(CORPUS)

### 4.2 — TF-IDF vs Semantic Search — Head-to-Head Comparison

Let's compare both engines on the same queries, paying special attention to cases where meaning matters more than keywords.

In [ ]:
comparison_queries = [
    "how do computers learn from data",              # Semantic: ML concepts
    "ancient trade between east and west",           # Semantic: Silk Road
    "protecting systems from hackers",               # Semantic: cybersecurity
    "machines that can think for themselves",         # Semantic: AI agents
    "clean energy from atoms",                       # Semantic: nuclear fusion
    "making copies of books before computers",       # Semantic: printing press
]

print("=" * 80)
print("TF-IDF vs SEMANTIC SEARCH COMPARISON")
print("=" * 80)

for query in comparison_queries:
    tfidf_results = tfidf_engine.search(query, top_k=3)
    semantic_results = semantic_engine.search(query, top_k=3)

    print(f"\nQuery: \"{query}\"")
    print(f"  TF-IDF top results:")
    for i, r in enumerate(tfidf_results[:3]):
        print(f"    {i+1}. {r['title']} ({r['score']:.4f})")
    if not tfidf_results:
        print(f"    (no results)")

    print(f"  Semantic top results:")
    for i, r in enumerate(semantic_results[:3]):
        print(f"    {i+1}. {r['title']} ({r['score']:.4f})")

    # Check if top result matches
    tfidf_top = tfidf_results[0]['title'] if tfidf_results else None
    semantic_top = semantic_results[0]['title'] if semantic_results else None
    match = "✓ AGREE" if tfidf_top == semantic_top else "✗ DISAGREE"
    print(f"  Top-1: {match}")

## 5. Content Extraction Tool

Search finds **which document** is relevant. But agents often need specific **passages** from within a document. The content extraction tool breaks documents into paragraphs and returns the most relevant ones.

In [ ]:
class ContentExtractor:
    """Extract relevant passages from a document given a query."""

    def extract(self, document, query, max_passages=3):
        """Split document into paragraphs, score each against query, return best."""
        paragraphs = [p.strip() for p in document['content'].split('\n\n') if p.strip()]

        if not paragraphs:
            return []

        # Embed query and paragraphs
        query_emb = embed_texts([query])
        para_embs = embed_texts(paragraphs)

        # Compute cosine similarity
        similarities = np.dot(para_embs, query_emb.T).flatten()

        # Rank paragraphs
        ranked = sorted(enumerate(similarities), key=lambda x: x[1], reverse=True)

        results = []
        for idx, sim in ranked[:max_passages]:
            results.append({
                "paragraph_index": idx,
                "text": paragraphs[idx],
                "relevance_score": round(float(sim), 4)
            })
        return results

extractor = ContentExtractor()

# Demo: extract relevant passages about "attention mechanism" from transformer article
doc = CORPUS[0]  # The Transformer Architecture Revolution
passages = extractor.extract(doc, "how does the attention mechanism work")

print(f"Document: {doc['title']}")
print(f"Query: 'how does the attention mechanism work'")
print(f"\nExtracted passages (ranked by relevance):")
for i, p in enumerate(passages):
    print(f"\n--- Passage {i+1} (relevance: {p['relevance_score']:.4f}) ---")
    print(p['text'])

## 6. Building a Research Agent

Now we combine everything into an agent that can **search → extract → synthesize**. Given a question, it:

1. Searches the corpus (tries both TF-IDF and semantic)
2. Extracts relevant passages from top documents
3. Synthesizes an answer with citations

In [ ]:
class ResearchAgent:
    """Agent that searches, extracts, and synthesizes answers from a corpus."""

    def __init__(self, tfidf_engine, semantic_engine, extractor):
        self.tfidf = tfidf_engine
        self.semantic = semantic_engine
        self.extractor = extractor
        self.search_log = []

    def research(self, question, verbose=True):
        """Answer a question using search + extraction + synthesis."""
        if verbose:
            print(f"\n{'='*70}")
            print(f"RESEARCH QUESTION: {question}")
            print(f"{'='*70}")

        # Step 1: Search with both engines
        tfidf_results = self.tfidf.search(question, top_k=3)
        semantic_results = self.semantic.search(question, top_k=3)

        # Merge results (union of top docs, prefer semantic scores)
        seen_ids = set()
        merged = []
        for r in semantic_results + tfidf_results:
            if r['doc_id'] not in seen_ids:
                seen_ids.add(r['doc_id'])
                merged.append(r)

        if verbose:
            print(f"\nStep 1 — Search Results ({len(merged)} unique documents):")
            for r in merged[:5]:
                print(f"  • [{r['topic']}] {r['title']} (score: {r['score']:.4f})")

        # Step 2: Extract relevant passages from top docs
        all_passages = []
        for r in merged[:3]:
            doc = CORPUS[r['doc_id']]
            passages = self.extractor.extract(doc, question, max_passages=2)
            for p in passages:
                p['source_title'] = doc['title']
                p['source_topic'] = doc['topic']
            all_passages.extend(passages)

        # Sort all passages by relevance
        all_passages.sort(key=lambda x: x['relevance_score'], reverse=True)
        top_passages = all_passages[:4]

        if verbose:
            print(f"\nStep 2 — Extracted {len(top_passages)} relevant passages")
            for p in top_passages:
                print(f"  • From \"{p['source_title']}\": {p['text'][:80]}...")

        # Step 3: Synthesize answer using LLM
        context = "\n\n".join([
            f"[Source: {p['source_title']}]\n{p['text']}" for p in top_passages
        ])

        messages = [
            {"role": "system", "content": (
                "You are a research assistant. Answer the question based ONLY on the "
                "provided sources. Cite sources using [Source: title] format. "
                "If the sources don't contain enough information, say so."
            )},
            {"role": "user", "content": (
                f"Question: {question}\n\n"
                f"Sources:\n{context}\n\n"
                f"Provide a comprehensive answer with citations."
            )}
        ]

        if verbose:
            print(f"\nStep 3 — Synthesizing answer...")

        answer = generate(messages, max_new_tokens=400)

        # Log the research process
        self.search_log.append({
            "question": question,
            "num_docs_found": len(merged),
            "num_passages_extracted": len(top_passages),
            "sources_used": [p['source_title'] for p in top_passages]
        })

        if verbose:
            print(f"\n--- ANSWER ---")
            print(answer)

        return answer

# Create the research agent
agent = ResearchAgent(tfidf_engine, semantic_engine, extractor)

In [ ]:
# Run research on diverse questions
research_questions = [
    "How does the transformer architecture differ from previous approaches to NLP?",
    "What is the current state of nuclear fusion research and when might it produce practical energy?",
    "How did the printing press change European society?",
    "What are the main ethical concerns with AI systems and what frameworks exist to address them?",
    "How do modern electric vehicle batteries compare to earlier technology in cost and performance?",
]

for q in research_questions:
    agent.research(q)
    print()

### 6.1 — Research Agent Analysis

Let's examine the agent's search patterns — which sources it found most useful and how well it handled different question types.

In [ ]:
print("\n" + "="*70)
print("RESEARCH AGENT — SEARCH LOG ANALYSIS")
print("="*70)

for i, log in enumerate(agent.search_log):
    print(f"\nQ{i+1}: {log['question'][:60]}...")
    print(f"  Documents found: {log['num_docs_found']}")
    print(f"  Passages extracted: {log['num_passages_extracted']}")
    print(f"  Sources used: {', '.join(set(log['sources_used']))}")

# Aggregate stats
all_sources = [s for log in agent.search_log for s in log['sources_used']]
source_counts = Counter(all_sources)
print(f"\n--- Most-cited sources ---")
for source, count in source_counts.most_common(5):
    print(f"  {count}x — {source}")

## 7. Multi-Query Search — Reformulating When Results Are Poor

Sometimes the initial query doesn't retrieve good results. A smarter agent **reformulates** the query — breaking it into sub-queries or rephrasing it — and combines results.

### The Strategy
1. Run initial search
2. If top scores are below a threshold, ask the LLM to generate alternative queries
3. Search with each alternative
4. Merge and deduplicate results

In [ ]:
class MultiQuerySearchAgent:
    """Agent that reformulates queries when initial search results are poor."""

    def __init__(self, semantic_engine, extractor, score_threshold=0.45):
        self.engine = semantic_engine
        self.extractor = extractor
        self.threshold = score_threshold

    def generate_alternative_queries(self, original_query):
        """Use LLM to generate reformulated queries."""
        messages = [
            {"role": "system", "content": (
                "Generate 3 alternative search queries for the given question. "
                "Each should approach the topic from a different angle. "
                "Return ONLY the queries, one per line, no numbering or explanation."
            )},
            {"role": "user", "content": original_query}
        ]
        response = generate(messages, max_new_tokens=150, temperature=0.7)
        alternatives = [q.strip() for q in response.strip().split('\n') if q.strip()]
        return alternatives[:3]

    def search(self, question, verbose=True):
        """Search with automatic query reformulation if needed."""
        if verbose:
            print(f"\n{'='*70}")
            print(f"MULTI-QUERY SEARCH: {question}")
            print(f"{'='*70}")

        # Initial search
        results = self.engine.search(question, top_k=5)
        top_score = results[0]['score'] if results else 0

        if verbose:
            print(f"\nInitial search — top score: {top_score:.4f} (threshold: {self.threshold})")
            for r in results[:3]:
                print(f"  • {r['title']} ({r['score']:.4f})")

        # Check if reformulation is needed
        if top_score >= self.threshold:
            if verbose:
                print(f"  ✓ Good results — no reformulation needed")
            return results

        # Generate alternative queries
        if verbose:
            print(f"  ✗ Below threshold — reformulating query...")

        alternatives = self.generate_alternative_queries(question)
        if verbose:
            print(f"  Alternative queries:")
            for alt in alternatives:
                print(f"    → {alt}")

        # Search with each alternative
        all_results = {r['doc_id']: r for r in results}  # Start with original results
        for alt_query in alternatives:
            alt_results = self.engine.search(alt_query, top_k=3)
            for r in alt_results:
                if r['doc_id'] not in all_results or r['score'] > all_results[r['doc_id']]['score']:
                    all_results[r['doc_id']] = r

        # Sort by best score
        merged = sorted(all_results.values(), key=lambda x: x['score'], reverse=True)

        if verbose:
            print(f"\nAfter reformulation — {len(merged)} unique results:")
            for r in merged[:5]:
                print(f"  • {r['title']} ({r['score']:.4f})")
            improvement = merged[0]['score'] - top_score if merged else 0
            print(f"  Score improvement: +{improvement:.4f}")

        return merged[:5]

mq_agent = MultiQuerySearchAgent(semantic_engine, extractor)

In [ ]:
# Test multi-query search with various queries — some easy, some challenging
multi_query_tests = [
    "how do stars make energy",                       # Indirect — maps to fusion
    "ways to protect private information online",     # Indirect — maps to cybersecurity
    "ancient methods of sharing knowledge",           # Indirect — printing press / silk road
    "building thinking machines",                     # Indirect — AI agents / computing history
]

for query in multi_query_tests:
    mq_agent.search(query)
    print()

## 8. Hybrid Search — Combining TF-IDF and Semantic

The best search systems use **both** keyword and semantic search. Let's build a hybrid that fuses scores from both engines.

In [ ]:
class HybridSearchEngine:
    """Combines TF-IDF and semantic search with score fusion."""

    def __init__(self, tfidf_engine, semantic_engine, semantic_weight=0.7):
        self.tfidf = tfidf_engine
        self.semantic = semantic_engine
        self.semantic_weight = semantic_weight
        self.tfidf_weight = 1 - semantic_weight

    def _normalize_scores(self, results):
        """Min-max normalize scores to [0, 1]."""
        if not results:
            return results
        scores = [r['score'] for r in results]
        min_s, max_s = min(scores), max(scores)
        range_s = max_s - min_s if max_s > min_s else 1
        for r in results:
            r['norm_score'] = (r['score'] - min_s) / range_s
        return results

    def search(self, query, top_k=5):
        """Hybrid search with weighted score fusion."""
        tfidf_results = self._normalize_scores(self.tfidf.search(query, top_k=10))
        semantic_results = self._normalize_scores(self.semantic.search(query, top_k=10))

        # Build score maps
        tfidf_scores = {r['doc_id']: r['norm_score'] for r in tfidf_results}
        semantic_scores = {r['doc_id']: r['norm_score'] for r in semantic_results}

        # Fuse scores for all docs that appear in either result set
        all_doc_ids = set(tfidf_scores.keys()) | set(semantic_scores.keys())
        fused = []
        for doc_id in all_doc_ids:
            ts = tfidf_scores.get(doc_id, 0.0)
            ss = semantic_scores.get(doc_id, 0.0)
            fused_score = self.tfidf_weight * ts + self.semantic_weight * ss
            fused.append({
                "doc_id": doc_id,
                "title": CORPUS[doc_id]["title"],
                "topic": CORPUS[doc_id]["topic"],
                "score": round(fused_score, 4),
                "tfidf_score": round(ts, 4),
                "semantic_score": round(ss, 4)
            })

        fused.sort(key=lambda x: x['score'], reverse=True)
        return fused[:top_k]

hybrid_engine = HybridSearchEngine(tfidf_engine, semantic_engine, semantic_weight=0.7)

# Compare all three engines
eval_queries = [
    "how transformers process language",
    "genetic modification technology",
    "communication networks across civilizations",
]

for query in eval_queries:
    print(f"\nQuery: \"{query}\"")
    hybrid_results = hybrid_engine.search(query, top_k=3)
    for r in hybrid_results:
        print(f"  {r['title']:45s}  hybrid={r['score']:.3f}  "
              f"(tfidf={r['tfidf_score']:.3f}, semantic={r['semantic_score']:.3f})")

## 9. Key Takeaways

### What We Built

| Component | Purpose | Technique |
|-----------|---------|-----------|
| **TF-IDF Engine** | Keyword search | Term frequency × inverse document frequency |
| **Semantic Engine** | Meaning-based search | Dense embeddings + FAISS |
| **Content Extractor** | Passage retrieval | Per-paragraph embedding similarity |
| **Research Agent** | Search + synthesize | Multi-source retrieval + LLM synthesis |
| **Multi-Query Agent** | Handle poor results | LLM-based query reformulation |
| **Hybrid Engine** | Best of both | Weighted score fusion |

### Key Insights

1. **TF-IDF is fast and interpretable** but misses semantic similarity ("car battery" ≠ "EV energy storage")
2. **Semantic search captures meaning** but can be misled by surface-level similarity
3. **Hybrid search combines strengths** — keywords for precision, embeddings for recall
4. **Multi-query reformulation** handles the "vocabulary mismatch" problem
5. **Content extraction** (passage-level retrieval) is more useful than whole-document retrieval
6. **The search → extract → synthesize pipeline** is the standard pattern for information retrieval agents

### Design Principles

- **Always cite sources** — agents that synthesize without attribution are unreliable
- **Use both search types** — hybrid consistently outperforms either alone
- **Score thresholds matter** — low scores mean "I don't know," not "here's a guess"
- **Log everything** — search logs enable debugging and improvement

---

**Next Notebook**: [16_file_and_data_tools.ipynb](./16_file_and_data_tools.ipynb) — building tools for file manipulation and data analysis.